# 03 · List and dict problems

Sixteen list and dictionary problems, from two-sum to deduplicating records, covering the hashing, two-pointer and counting moves that turn O(n²) answers into O(n). Afterwards you can write each one idiomatically, say its complexity, and name the trap the interviewer is waiting for.

**Time:** ~50 min · **Runs:** offline on CPU in < 15 s · **Needs:** [Lists and tuples](../01_python_basics/03_lists_and_tuples.ipynb), [Dicts and sets](../01_python_basics/04_dicts_and_sets.ipynb), [01 · Warm-up problems](01_warmup_problems.ipynb)

## Why this matters in interviews

- "Use a dict" is the single most common optimisation in coding screens: two-sum, grouping, counting and deduping all go from O(n²) to O(n) with a hash map.
- Production Python (configs, API payloads, logs, LLM tool results) is lists of dicts. Safe nested access, merging, inverting and deduping records are everyday tasks, and interviewers know it.
- The follow-ups test Python itself: stable sorting, dict ordering, `Counter` arithmetic, lazy generators, in-place vs copy.

## What interviewers ask

- "Two-sum: return the indices of two numbers that add up to the target." Then: "better than O(n²)?"
- "Group these words into anagrams" / "give me the top-k most frequent words."
- "Rotate a list by k", "move the zeros to the end in place", "merge two sorted lists."
- "Split this list into batches of 100 for an API."
- "Print a matrix in spiral order."
- "Invert this dict. What if two keys share a value?"
- "Get `config['db']['primary']['host']` without crashing when a level is missing."
- "Deduplicate these records by email, keeping the latest."

Each problem has a statement, a solution cell, a test cell (`assert`s: silence means pass) and one line on complexity. To practise, write your own version and add it to the tuple the test cell loops over.

In [ ]:
import heapq
import random
import time
from collections import Counter, defaultdict, deque
from itertools import accumulate, batched, count, islice

import matplotlib.pyplot as plt

random.seed(0)
plt.rcParams.update({                      # quiet chart style; colour-blind-safe colours below
    "axes.spines.top": False, "axes.spines.right": False, "axes.edgecolor": "#c3c2b7",
    "axes.grid": True, "grid.color": "#e1e0d9", "grid.linewidth": 0.8,
    "axes.axisbelow": True, "axes.titlesize": 11, "legend.frameon": False,
})
BLUE, ORANGE = "#2a78d6", "#eb6834"


def raises(exc_type, fn, *args, **kwargs) -> bool:
    """True if fn(*args) raises exc_type (a tiny stand-in for pytest.raises)."""
    try:
        fn(*args, **kwargs)
    except exc_type:
        return True
    return False

## 1 · Arrays and two pointers

### Problem 1 — Two-sum (hash map)

Return indices `(i, j)`, `i < j`, of two numbers that add up to `target`, or `None`.
Example: `two_sum([2, 7, 11, 15], 9)` → `(0, 1)`.

**In plain English:** for each number, the partner you need is `target - x`. Remember every number you have passed in a dict, and each lookup is O(1).

In [ ]:
def two_sum_brute(nums, target):
    for i in range(len(nums)):
        for j in range(i + 1, len(nums)):          # every pair once: O(n²)
            if nums[i] + nums[j] == target:
                return i, j
    return None


def two_sum(nums, target):
    seen = {}                                      # value -> index where we saw it
    for j, x in enumerate(nums):
        i = seen.get(target - x)                   # have we already passed the partner?
        if i is not None:
            return i, j
        seen[x] = j                                # store AFTER checking, so x can't pair with itself
    return None


print(two_sum([2, 7, 11, 15], 9), two_sum([3, 3], 6), two_sum([1, 2, 3], 7))

In [ ]:
for f in (two_sum_brute, two_sum):
    assert f([], 5) is None and f([5], 10) is None     # empty; single (can't use one element twice)
    assert f([2, 7, 11, 15], 9) == (0, 1)
    assert f([3, 3], 6) == (0, 1)                      # duplicates
    assert f([-3, 4, 3, 90], 0) == (0, 2)              # negatives
    assert f([0, 4, 3, 0], 0) == (0, 3)                # zeros
    assert f([1, 2, 3], 7) is None                     # no solution
for _ in range(500):                                   # property test against the brute force
    nums = [random.randint(-10, 10) for _ in range(random.randint(0, 8))]
    target = random.randint(-15, 15)
    fast, slow = two_sum(nums, target), two_sum_brute(nums, target)
    assert (fast is None) == (slow is None)
    if fast:
        i, j = fast
        assert i < j and nums[i] + nums[j] == target
print("two_sum: all tests passed")

In [ ]:
sizes = [500, 1000, 2000, 3000]
brute_ms, hash_ms = [], []
for n in sizes:
    nums = list(range(0, 2 * n, 2))                   # all even: an odd target has no answer (worst case)
    t0 = time.perf_counter(); assert two_sum_brute(nums, -1) is None
    brute_ms.append((time.perf_counter() - t0) * 1000)
    t0 = time.perf_counter(); assert two_sum(nums, -1) is None
    hash_ms.append((time.perf_counter() - t0) * 1000)
print("brute ms   :", [round(t, 1) for t in brute_ms])
print("hash map ms:", [round(t, 2) for t in hash_ms])
assert brute_ms[-1] > 10 * hash_ms[-1]

fig, ax = plt.subplots(figsize=(9, 3.6))
ax.plot(sizes, brute_ms, color=ORANGE, lw=2, marker="o", label="brute force: check every pair")
ax.plot(sizes, hash_ms, color=BLUE, lw=2, marker="o", label="hash map: one pass")
ax.set_xlabel("list length (no pair adds up: the worst case)"); ax.set_ylabel("milliseconds")
ax.set_title("Two-sum: O(n²) pairs vs one O(n) pass with a dict")
ax.legend(loc="upper left")
plt.show()

**Complexity:** O(n) time and O(n) space with the dict, versus O(n²) time and O(1) space. If the list is sorted, two pointers from both ends give O(n) time with O(1) space. Clarify: return indices or values? Are there several answers?

### Problem 2 — Rotate a list by k

Rotate right by k; k may exceed the length or be negative (rotate left).
Example: `rotate([1, 2, 3, 4, 5], 2)` → `[4, 5, 1, 2, 3]`.

In [ ]:
def rotate(nums, k):
    if not nums:
        return []
    k %= len(nums)                                # k > n and negative k both reduce to 0..n-1
    return nums[-k:] + nums[:-k]                  # k == 0: nums[0:] + nums[:0], a full copy


def rotate_in_place(nums, k):
    """O(1) extra space: reverse everything, then the first k, then the rest."""
    n = len(nums)
    if n == 0:
        return
    k %= n

    def reverse(i, j):
        while i < j:
            nums[i], nums[j] = nums[j], nums[i]
            i, j = i + 1, j - 1

    reverse(0, n - 1)
    reverse(0, k - 1)
    reverse(k, n - 1)


def rotate_deque(nums, k):
    d = deque(nums)
    d.rotate(k)                                   # deque handles k > n and negative k
    return list(d)


def rotate_via_in_place(nums, k):                 # adapter so the tests can compare all three
    out = list(nums)
    rotate_in_place(out, k)
    return out


print(rotate([1, 2, 3, 4, 5], 2), rotate([1, 2, 3, 4, 5], -1))

In [ ]:
for f in (rotate, rotate_via_in_place, rotate_deque):
    assert f([], 3) == [] and f([1], 3) == [1]            # empty, single
    assert f([1, 2, 3, 4, 5], 2) == [4, 5, 1, 2, 3]
    assert f([1, 2, 3, 4, 5], 7) == [4, 5, 1, 2, 3]       # k > n
    assert f([1, 2, 3, 4, 5], -1) == [2, 3, 4, 5, 1]      # negative k rotates left
    assert f([1, 2, 3], 0) == [1, 2, 3]
    assert f([1, 1, 2], 1) == [2, 1, 1]                   # duplicates
data = list(range(7))
assert all(rotate(data, k) == rotate_via_in_place(data, k) == rotate_deque(data, k) for k in range(-10, 11))
assert rotate(data, 3) is not data and data == list(range(7))   # slicing never modifies the input
print("rotate: all tests passed")

**Complexity:** slicing is O(n) time and O(n) space; the three-reversal trick is O(n) time and O(1) extra space. `deque.rotate` is the right tool when you rotate repeatedly.

### Problem 3 — Move zeros to the end (in place)

Keep the order of the non-zero elements. Example: `[0, 1, 0, 3, 12]` → `[1, 3, 12, 0, 0]`.

In [ ]:
def move_zeros(nums):
    """In place and stable: `write` is where the next non-zero belongs."""
    write = 0
    for read in range(len(nums)):
        if nums[read] != 0:
            nums[write], nums[read] = nums[read], nums[write]
            write += 1
    return nums


def move_zeros_copy(nums):
    return [x for x in nums if x != 0] + [x for x in nums if x == 0]


def move_zeros_sort(nums):
    return sorted(nums, key=lambda x: x == 0)     # False < True, and sorted() is stable


print(move_zeros([0, 1, 0, 3, 12]))

In [ ]:
for f in (move_zeros, move_zeros_copy, move_zeros_sort):
    assert f([]) == [] and f([0]) == [0] and f([1]) == [1]   # empty, single
    assert f([0, 1, 0, 3, 12]) == [1, 3, 12, 0, 0]
    assert f([0, 0, 1]) == [1, 0, 0] and f([0, 0]) == [0, 0]
    assert f([-1, 0, -2]) == [-1, -2, 0]                     # negatives keep their order
    assert f([4, 2, 0, 1]) == [4, 2, 1, 0]                   # non-zeros keep order (not sorted)
lst = [0, 1]
assert move_zeros(lst) is lst and lst == [1, 0]              # truly in place
print("move_zeros: all tests passed")

**Complexity:** the write-pointer version is O(n) time and O(1) extra space; the copy is O(n) space; the sort trick is O(n log n). Watch out for `False == 0`: a `False` in the list counts as a zero.

### Problem 4 — Merge two sorted lists

Example: `merge_sorted([1, 3, 5], [2, 3, 6])` → `[1, 2, 3, 3, 5, 6]` in linear time.

In [ ]:
def merge_sorted(a, b):
    i = j = 0
    out = []
    while i < len(a) and j < len(b):
        if a[i] <= b[j]:                          # <= keeps it stable: ties take from a first
            out.append(a[i]); i += 1
        else:
            out.append(b[j]); j += 1
    out.extend(a[i:])                             # at most one of these is non-empty
    out.extend(b[j:])
    return out


print(merge_sorted([1, 3, 5], [2, 3, 6]), list(heapq.merge([1, 3, 5], [2, 3, 6])))

In [ ]:
assert merge_sorted([], []) == []                                     # both empty
assert merge_sorted([], [1]) == [1] and merge_sorted([1], []) == [1]  # one empty
assert merge_sorted([1, 3, 5], [2, 3, 6]) == [1, 2, 3, 3, 5, 6]       # duplicates
assert merge_sorted([-5, -1], [-3, 0]) == [-5, -3, -1, 0]             # negatives
assert merge_sorted([1, 2], [3, 4]) == [1, 2, 3, 4]                   # no interleaving
for _ in range(300):
    a = sorted(random.randint(-20, 20) for _ in range(random.randint(0, 8)))
    b = sorted(random.randint(-20, 20) for _ in range(random.randint(0, 8)))
    assert merge_sorted(a, b) == sorted(a + b) == list(heapq.merge(a, b))
print("merge_sorted: all tests passed")

**Complexity:** O(n + m) time and space. `heapq.merge` does the same lazily for any number of sorted inputs (the k-way merge behind external sorting). `sorted(a + b)` is also fast in CPython because its sort detects the two pre-sorted runs.

### Problem 5 — Running sum (prefix sums)

Example: `[1, 2, 3, 4]` → `[1, 3, 6, 10]`. Then use prefix sums to answer "sum of `nums[i:j]`" in O(1).

In [ ]:
def running_sum_loop(nums):
    out, total = [], 0
    for x in nums:
        total += x
        out.append(total)
    return out


def running_sum(nums):
    return list(accumulate(nums))


def range_sum(prefix, i, j):
    """sum(nums[i:j]) in O(1), given prefix = [0, *running_sum(nums)]."""
    return prefix[j] - prefix[i]


nums = [3, 1, 4, 1, 5, 9, 2, 6]
prefix = list(accumulate(nums, initial=0))
print(running_sum(nums), "| sum(nums[2:5]) =", range_sum(prefix, 2, 5))

In [ ]:
for f in (running_sum_loop, running_sum):
    assert f([]) == [] and f([5]) == [5]                  # empty, single
    assert f([1, 2, 3, 4]) == [1, 3, 6, 10]
    assert f([3, -3, 3]) == [3, 0, 3]                     # negatives
    assert f([0.5, 0.25]) == [0.5, 0.75]                  # floats
data = [random.randint(-9, 9) for _ in range(40)]
pre = list(accumulate(data, initial=0))
assert all(range_sum(pre, i, j) == sum(data[i:j]) for i in range(41) for j in range(i, 41))
print("running_sum: all tests passed")

**Complexity:** O(n) to build, then O(1) per range query instead of O(n). The same idea powers "subarray sum equals k": count prefix sums in a dict.

## 2 · Batching and matrices

### Problem 6 — Chunk a list into batches of n

Example: `chunk([1, 2, 3, 4, 5, 6, 7], 3)` → `[[1, 2, 3], [4, 5, 6], [7]]`. Also write a lazy version that works on any iterable (a file, a generator, a database cursor).

In [ ]:
def chunk(items, n):
    if n < 1:
        raise ValueError("batch size must be >= 1")
    return [items[i:i + n] for i in range(0, len(items), n)]


def ichunk(iterable, n):
    """Lazy batches from ANY iterable, holding only one batch in memory."""
    if n < 1:
        raise ValueError("batch size must be >= 1")
    it = iter(iterable)
    while batch := list(islice(it, n)):           # up to n items; [] means the source is exhausted
        yield batch


print(chunk(list(range(1, 8)), 3), list(ichunk("abcdefg", 3)), list(batched(range(1, 8), 3)))

In [ ]:
for f in (chunk, lambda xs, n: list(ichunk(xs, n)), lambda xs, n: [list(b) for b in batched(xs, n)]):
    assert f([], 3) == [] and f([1], 3) == [[1]]                      # empty, single
    assert f(list(range(1, 8)), 3) == [[1, 2, 3], [4, 5, 6], [7]]     # ragged last batch
    assert f(list(range(6)), 3) == [[0, 1, 2], [3, 4, 5]]             # exact multiple
    assert f([1, 2], 5) == [[1, 2]]                                   # n bigger than the list
    assert f([1, 2, 3], 1) == [[1], [2], [3]]
data = list(range(23))
assert [x for b in chunk(data, 4) for x in b] == data                 # nothing lost, order kept
assert list(islice(ichunk(count(), 4), 2)) == [[0, 1, 2, 3], [4, 5, 6, 7]]   # works on an infinite stream
assert raises(ValueError, chunk, [1], 0) and raises(ValueError, batched, [1], 0)
gen = ichunk([1], 0)                                  # no error yet: a generator's body runs lazily...
assert raises(ValueError, next, gen)                  # ...so the check fires on the first next()
print("chunk: all tests passed")

**Complexity:** O(n) time. The slicing version copies the whole list; the generator holds one batch at a time, which is what you want for an embeddings pipeline. Python 3.12 has `itertools.batched` (it yields tuples). Gotcha: validation inside a generator runs on the first `next()`, not at the call.

### Problem 7 — Matrix transpose and spiral order

Transpose: `[[1, 2, 3], [4, 5, 6]]` → `[[1, 4], [2, 5], [3, 6]]`.
Spiral: `[[1, 2, 3, 4], [5, 6, 7, 8], [9, 10, 11, 12]]` → `[1, 2, 3, 4, 8, 12, 11, 10, 9, 5, 6, 7]`.

In [ ]:
def transpose(m):
    return [list(row) for row in zip(*m)]         # zip(*m) pairs up the i-th item of every row


def spiral_order(m):
    out = []
    if not m or not m[0]:
        return out
    top, bottom, left, right = 0, len(m) - 1, 0, len(m[0]) - 1
    while top <= bottom and left <= right:
        out.extend(m[top][c] for c in range(left, right + 1)); top += 1        # → top row
        out.extend(m[r][right] for r in range(top, bottom + 1)); right -= 1    # ↓ right column
        if top <= bottom:
            out.extend(m[bottom][c] for c in range(right, left - 1, -1)); bottom -= 1   # ← bottom row
        if left <= right:
            out.extend(m[r][left] for r in range(bottom, top - 1, -1)); left += 1       # ↑ left column
    return out


def spiral_rotate(m):
    """Peel off the top row, rotate the rest counter-clockwise, repeat. Short, but copies each layer."""
    out, rest = [], [list(row) for row in m]
    while rest:
        out += rest.pop(0)
        rest = [list(col) for col in zip(*rest)][::-1]
    return out


grid = [[1, 2, 3, 4], [5, 6, 7, 8], [9, 10, 11, 12]]
print(transpose([[1, 2, 3], [4, 5, 6]]), spiral_order(grid))

In [ ]:
assert transpose([]) == [] and transpose([[1]]) == [[1]]              # empty, single
assert transpose([[1, 2, 3]]) == [[1], [2], [3]]
assert transpose([[1, 2, 3], [4, 5, 6]]) == [[1, 4], [2, 5], [3, 6]]
assert transpose(transpose(grid)) == grid                             # transposing twice is identity
assert transpose([[1, 2], [3]]) == [[1, 3]]                           # trap: zip silently truncates ragged rows
assert raises(ValueError, lambda: list(zip(*[[1, 2], [3]], strict=True)))   # strict=True catches it
for f in (spiral_order, spiral_rotate):
    assert f([]) == [] and f([[]]) == [] and f([[7]]) == [7]          # empty, empty row, single
    assert f([[1, 2, 3]]) == [1, 2, 3] and f([[1], [2], [3]]) == [1, 2, 3]   # one row / one column
    assert f([[1, 2], [3, 4]]) == [1, 2, 4, 3]
    assert f(grid) == [1, 2, 3, 4, 8, 12, 11, 10, 9, 5, 6, 7]
    assert f([[-1, -2], [-3, -4], [-5, -6]]) == [-1, -2, -4, -6, -5, -3]   # tall matrix, negatives
for _ in range(100):                                                  # both versions agree on random shapes
    r, c = random.randint(1, 6), random.randint(1, 6)
    m = [[random.randint(0, 9) for _ in range(c)] for _ in range(r)]
    assert spiral_order(m) == spiral_rotate(m) and sorted(spiral_order(m)) == sorted(sum(m, []))
print("transpose/spiral: all tests passed")

The picture shows the visit order on a 5 × 6 grid: the boundary version walks the outer ring, shrinks the four boundaries by one, and repeats.

In [ ]:
R, C = 5, 6
path = spiral_order([[(r, c) for c in range(C)] for r in range(R)])    # the cells, in visit order
order = [[0] * C for _ in range(R)]
for step, (r, c) in enumerate(path, start=1):
    order[r][c] = step

fig, ax = plt.subplots(figsize=(6.5, 4.4))
ax.imshow(order, cmap="Blues", vmin=-6, vmax=R * C + 4)
for r in range(R):
    for c in range(C):
        ax.text(c, r, order[r][c], ha="center", va="center", fontsize=10,
                color="white" if order[r][c] > 18 else "#0b0b0b")
ys, xs = zip(*path)
ax.plot(xs, ys, color=ORANGE, lw=1.6, alpha=0.9)
ax.set_xticks(range(C)); ax.set_yticks(range(R)); ax.grid(False)
ax.set_xlabel("column"); ax.set_ylabel("row")
ax.set_title("Spiral order: outer ring first, then inwards")
plt.show()

**Complexity:** transpose is O(r·c). The boundary spiral is O(r·c) time and O(1) extra space besides the output; the rotate version is shorter but re-copies the shrinking matrix each layer, so it does more work (fine to mention, fine to write if the interviewer wants brevity).

## 3 · Counting and hashing

### Problem 8 — Group anagrams

Example: `["eat", "tea", "tan", "ate", "nat", "bat"]` → `[["eat", "tea", "ate"], ["tan", "nat"], ["bat"]]`.

**In plain English:** anagrams share the same letters, so give each word a canonical key (its letters sorted) and collect words by key.

In [ ]:
def group_anagrams(words):
    groups = defaultdict(list)
    for w in words:
        groups["".join(sorted(w))].append(w)           # key: the letters in sorted order
    return list(groups.values())                       # groups appear in first-seen order


def group_anagrams_counts(words):
    """O(L) key per word instead of O(L log L), but only valid for lowercase a-z."""
    groups = defaultdict(list)
    for w in words:
        counts = [0] * 26
        for ch in w:
            counts[ord(ch) - ord("a")] += 1
        groups[tuple(counts)].append(w)                # lists aren't hashable; tuples are
    return list(groups.values())


print(group_anagrams(["eat", "tea", "tan", "ate", "nat", "bat"]))

In [ ]:
example = ["eat", "tea", "tan", "ate", "nat", "bat"]
for f in (group_anagrams, group_anagrams_counts):
    assert f([]) == []                                          # empty
    assert f([""]) == [[""]] and f(["a"]) == [["a"]]            # empty word, single
    assert f(example) == [["eat", "tea", "ate"], ["tan", "nat"], ["bat"]]
    assert f(["ab", "ba", "ab"]) == [["ab", "ba", "ab"]]        # duplicates stay
assert group_anagrams(["résumé", "mérésu"]) == [["résumé", "mérésu"]]   # sorted keys handle unicode...
assert group_anagrams_counts(["Z", "t"]) == [["Z", "t"]]        # ...the 26-slot key is WRONG here:
                                                                # ord('Z') - ord('a') = -7 wraps to slot 19 ('t')
print("group_anagrams: all tests passed")

**Complexity:** O(N · L log L) with sorted keys, O(N · L) with count keys, for N words of length L. The count-key trick needs input you control: uppercase letters silently land in the wrong slot because negative indexes wrap around.

### Problem 9 — Top-k frequent words

Most frequent first; ties alphabetical.
Example: `top_k(["i", "love", "code", "i", "love", "coding"], 2)` → `["i", "love"]`.

In [ ]:
def top_k_sort(words, k):
    counts = Counter(words)
    return sorted(counts, key=lambda w: (-counts[w], w))[:k]        # O(m log m)


def top_k_heap(words, k):
    counts = Counter(words)
    return heapq.nsmallest(k, counts, key=lambda w: (-counts[w], w))   # O(m log k): only k in the heap


print(top_k_heap(["i", "love", "code", "i", "love", "coding"], 2))

In [ ]:
for f in (top_k_sort, top_k_heap):
    assert f([], 3) == [] and f(["a", "b"], 0) == []            # empty input, k = 0
    assert f(["x"], 5) == ["x"]                                 # single; k bigger than distinct words
    assert f(["i", "love", "code", "i", "love", "coding"], 2) == ["i", "love"]
    assert f(["b", "a", "b", "a", "c"], 2) == ["a", "b"]        # tie on count -> alphabetical
    assert f(["Word", "word", "word"], 2) == ["word", "Word"]   # case-sensitive unless you fold
for _ in range(200):
    words = [random.choice("abcdef") for _ in range(random.randint(0, 30))]
    k = random.randint(0, 7)
    assert top_k_sort(words, k) == top_k_heap(words, k)
print("top_k: all tests passed")

**Complexity:** counting is O(n); then O(m log m) to sort m distinct words or O(m log k) with a heap. Bucket sort by count gives O(n) overall. `Counter.most_common(k)` is fine too, but its ties follow first-seen order, not alphabetical.

### Problem 10 — Intersection of two lists, with counts

Each common element appears as many times as it occurs in *both*.
Example: `[1, 2, 2, 1]` ∩ `[2, 2]` → `[2, 2]`; `[4, 9, 5]` ∩ `[9, 4, 9, 8, 4]` → `[4, 9]`.

In [ ]:
def intersect(a, b):
    return list((Counter(a) & Counter(b)).elements())    # & keeps the minimum count of each key


def intersect_sorted(a, b):
    """Two pointers over sorted copies: O(1) extra space if the inputs are already sorted."""
    a, b = sorted(a), sorted(b)
    i = j = 0
    out = []
    while i < len(a) and j < len(b):
        if a[i] == b[j]:
            out.append(a[i]); i += 1; j += 1
        elif a[i] < b[j]:
            i += 1
        else:
            j += 1
    return out


print(intersect([1, 2, 2, 1], [2, 2]), intersect([4, 9, 5], [9, 4, 9, 8, 4]), set([1, 2, 2, 1]) & set([2, 2]))

In [ ]:
for f in (intersect, intersect_sorted):
    assert f([], [1]) == [] and f([1], []) == []                     # empty
    assert f([7], [7]) == [7]                                        # single
    assert sorted(f([1, 2, 2, 1], [2, 2])) == [2, 2]                 # duplicates are kept
    assert sorted(f([4, 9, 5], [9, 4, 9, 8, 4])) == [4, 9]
    assert sorted(f([-1, -1, 3], [-1, 3, 3])) == [-1, 3]             # negatives
    assert f([1, 2], [3, 4]) == []                                   # nothing in common
for _ in range(300):
    a = [random.randint(-3, 3) for _ in range(random.randint(0, 10))]
    b = [random.randint(-3, 3) for _ in range(random.randint(0, 10))]
    assert sorted(intersect(a, b)) == intersect_sorted(a, b)
assert set([1, 2, 2, 1]) & set([2, 2]) == {2}                         # sets lose the counts
print("intersect: all tests passed")

**Complexity:** O(n + m) time with `Counter`; O(n log n + m log m) with sorting (O(n + m) if already sorted). If one list is huge and on disk, count the small one and stream the big one.

### Problem 11 — Find duplicates

Return each value that appears more than once, in order of first appearance.
Example: `[3, 1, 3, 2, 1, 3]` → `[3, 1]`.

In [ ]:
def find_duplicates_brute(items):
    out = []
    for i, x in enumerate(items):
        if x in items[i + 1:] and x not in out:       # two linear scans per item: O(n²)
            out.append(x)
    return out


def find_duplicates(items):
    counts = Counter(items)                          # a Counter keeps first-seen key order
    return [x for x, c in counts.items() if c > 1]


print(find_duplicates([3, 1, 3, 2, 1, 3]))

In [ ]:
for f in (find_duplicates_brute, find_duplicates):
    assert f([]) == [] and f([1]) == []                  # empty, single
    assert f([1, 1]) == [1]
    assert f([3, 1, 3, 2, 1, 3]) == [3, 1]               # each duplicate reported once
    assert f([-1, 2, -1]) == [-1]                        # negatives
    assert f(["b", "a", "b", "a"]) == ["b", "a"]         # strings, first-seen order
    assert f([1, 2, 3]) == []                            # no duplicates
for _ in range(300):
    data = [random.randint(0, 6) for _ in range(random.randint(0, 12))]
    assert find_duplicates(data) == find_duplicates_brute(data)
print("find_duplicates: all tests passed")

**Complexity:** O(n) time and space with a `Counter` (or a `seen` set), versus O(n²). Clarify the order you want: first appearance or first *repeat* (for `[1, 2, 2, 1]` those give `[1, 2]` and `[2, 1]`). For "values 1..n in a list of length n" there is an O(1)-space trick that marks visited indexes by negating them.

## 4 · Dictionaries

### Problem 12 — Invert a dict (duplicate values)

Example: `{"a": 1, "b": 2, "c": 1}` → `{1: ["a", "c"], 2: ["b"]}`. The naive `{v: k}` silently loses keys.

In [ ]:
def invert_naive(d):
    return {v: k for k, v in d.items()}           # later keys overwrite earlier ones!


def invert(d):
    out = defaultdict(list)
    for k, v in d.items():
        out[v].append(k)
    return dict(out)


def invert_unique(d):
    inv = {v: k for k, v in d.items()}
    if len(inv) != len(d):
        raise ValueError("values are not unique; use invert() to group keys")
    return inv


d = {"a": 1, "b": 2, "c": 1}
print(invert(d), "| naive:", invert_naive(d))

In [ ]:
assert invert({}) == {} and invert({"a": 1}) == {1: ["a"]}           # empty, single
assert invert({"a": 1, "b": 2, "c": 1}) == {1: ["a", "c"], 2: ["b"]}  # duplicates grouped
assert invert_naive({"a": 1, "b": 2, "c": 1}) == {1: "c", 2: "b"}     # the bug: "a" vanished
assert raises(ValueError, invert_unique, {"a": 1, "b": 1})
ids = {"alice": 7, "bob": -3, "zoë": 12}                              # negatives, unicode keys
assert invert_unique(invert_unique(ids)) == ids                       # round trip when values are unique
assert raises(TypeError, invert, {"a": [1, 2]})                       # list values are unhashable keys
print("invert: all tests passed")

**Complexity:** O(n) time and space. Ask first: are values unique, and are they hashable? The silent overwrite is the bug the question is testing for.

### Problem 13 — Merge dicts, summing values

Example: `{"a": 1, "b": 2}` + `{"b": 3, "c": 4}` → `{"a": 1, "b": 5, "c": 4}`, for any number of dicts.

In [ ]:
def merge_sum(*dicts):
    out = {}
    for d in dicts:
        for k, v in d.items():
            out[k] = out.get(k, 0) + v
    return out


def merge_sum_counter(*dicts):
    total = Counter()
    for d in dicts:
        total.update(d)               # Counter.update ADDS; dict.update would replace
    return dict(total)


print(merge_sum({"a": 1, "b": 2}, {"b": 3, "c": 4}), "| `|` overwrites:", {"a": 1, "b": 2} | {"b": 3, "c": 4})

In [ ]:
for f in (merge_sum, merge_sum_counter):
    assert f() == {} and f({"a": 1}) == {"a": 1}                        # no dicts, one dict
    assert f({"a": 1, "b": 2}, {"b": 3, "c": 4}) == {"a": 1, "b": 5, "c": 4}
    assert f({"x": 1}, {"x": -1}) == {"x": 0}                           # zero results are kept
    assert f({"t": 0.5}, {"t": 0.25}, {"t": 1}) == {"t": 1.75}          # floats, many dicts
assert dict(Counter({"x": 1}) + Counter({"x": -1})) == {}               # trap: Counter + drops counts <= 0
assert ({"b": 2} | {"b": 3}) == {"b": 3}                                # `|` merges by overwriting
print("merge_sum: all tests passed")

**Complexity:** O(total number of keys). Know the three merges: `a | b` overwrites, `Counter.update` adds, and `Counter + Counter` adds but drops non-positive results.

### Problem 14 — Sort a dict by value, then by key

Values descending, ties by key ascending; return a dict (insertion-ordered).
Example: `{"b": 2, "a": 2, "c": 5, "d": 1}` → `{"c": 5, "a": 2, "b": 2, "d": 1}`.

In [ ]:
def sort_by_value_then_key(d):
    return dict(sorted(d.items(), key=lambda kv: (-kv[1], kv[0])))   # negation: numbers only


def sort_two_pass(d):
    """Works for any comparable values: sort by the secondary key, then stably by the primary."""
    items = sorted(d.items(), key=lambda kv: kv[0])                    # secondary: key ascending
    items.sort(key=lambda kv: kv[1], reverse=True)                     # primary: value descending
    return dict(items)                                                 # reverse=True stays stable


print(sort_by_value_then_key({"b": 2, "a": 2, "c": 5, "d": 1}))

In [ ]:
for f in (sort_by_value_then_key, sort_two_pass):
    assert f({}) == {} and list(f({"a": 1}).items()) == [("a", 1)]    # empty, single
    got = f({"b": 2, "a": 2, "c": 5, "d": 1})
    assert list(got.items()) == [("c", 5), ("a", 2), ("b", 2), ("d", 1)]   # compare ORDER, not just contents
    assert list(f({"x": -1, "y": -5, "z": 0})) == ["z", "x", "y"]           # negatives
assert list(sort_two_pass({"x": "b", "y": "a", "z": "b"})) == ["x", "z", "y"]   # string values
assert raises(TypeError, sort_by_value_then_key, {"x": "b"})               # can't negate a string
assert {"a": 1, "b": 2} == {"b": 2, "a": 1}                                # dict == ignores order!
print("sort dict: all tests passed")

**Complexity:** O(n log n). Two traps: `==` on dicts ignores order, so test with `list(d.items())`; and negating the value only works for numbers. The two-pass stable sort works for anything comparable.

### Problem 15 — Safe nested-dict get by dotted path

`get_path(cfg, "db.primary.host")` → `"10.0.0.1"`; a missing level returns the default; list indexes work: `"servers.0.name"`.

In [ ]:
CFG = {
    "db": {"primary": {"host": "10.0.0.1", "port": 5432}},
    "servers": [{"name": "a"}, {"name": "b"}],
    "flags": {"beta": False, "retries": 0},
    "café": {"menü": "ok"},
}


def get_path(data, path, default=None, sep="."):
    """Look before you leap: check each level's type and bounds."""
    cur = data
    for part in path.split(sep) if path else []:
        if isinstance(cur, dict):
            if part not in cur:
                return default
            cur = cur[part]
        elif isinstance(cur, list) and part.lstrip("-").isdigit():
            idx = int(part)
            if not -len(cur) <= idx < len(cur):
                return default
            cur = cur[idx]
        else:
            return default                     # a str/int/None can't be traversed further
    return cur


def get_path_eafp(data, path, default=None, sep="."):
    """Easier to ask forgiveness than permission: try it, catch what can go wrong."""
    if not path:
        return data
    try:
        for part in path.split(sep):
            data = data[int(part)] if isinstance(data, list) else data[part]
        return data
    except (KeyError, IndexError, TypeError, ValueError):
        return default


print(get_path(CFG, "db.primary.host"), get_path(CFG, "servers.-1.name"), get_path(CFG, "db.replica.host", "n/a"))

In [ ]:
for f in (get_path, get_path_eafp):
    assert f(CFG, "") == CFG                                        # empty path: the whole thing
    assert f(CFG, "db.primary.host") == "10.0.0.1" and f(CFG, "db.primary.port") == 5432
    assert f(CFG, "servers.1.name") == "b" and f(CFG, "servers.-1.name") == "b"   # list indexes, negative too
    assert f(CFG, "servers.5.name", "?") == "?"                     # index out of range
    assert f(CFG, "servers.x", "?") == "?"                          # non-numeric index into a list
    assert f(CFG, "db.replica.host", "?") == "?"                    # missing level
    assert f(CFG, "db.primary.host.x", "?") == "?"                  # can't go inside a string
    assert f(CFG, "flags.beta", True) is False                      # falsy values are real values
    assert f(CFG, "flags.retries", 3) == 0
    assert f(CFG, "café.menü") == "ok"                              # unicode keys
naive = CFG.get("flags", {}).get("retries") or 3                    # the classic `or` bug
assert naive == 3                                                   # ...0 was a real value, now it's 3
print("get_path: all tests passed")

**Complexity:** O(depth). Don't chain `.get(...) or default` (it replaces legitimate `0`, `""` and `False`). Keys that contain the separator need a list-of-keys API instead of a dotted string. Both LBYL and EAFP are Pythonic; EAFP is shorter, while LBYL makes each failure explicit.

### Problem 16 — Deduplicate a list of dicts by a key

Keep the first record per `id` (or the last). Dicts are unhashable, so `set(records)` fails.
Example: `[{"id": 1, "v": "a"}, {"id": 2, "v": "b"}, {"id": 1, "v": "c"}]` → first: `a, b`; last: `c, b`.

In [ ]:
def dedupe_by(records, key, keep="first"):
    get = key if callable(key) else (lambda r: r[key])       # a field name or a function
    if keep == "first":
        seen, out = set(), []
        for r in records:
            k = get(r)
            if k not in seen:
                seen.add(k)
                out.append(r)
        return out
    if keep == "last":
        return list({get(r): r for r in records}.values())    # later records overwrite earlier ones
    raise ValueError("keep must be 'first' or 'last'")


rows = [{"id": 1, "v": "a"}, {"id": 2, "v": "b"}, {"id": 1, "v": "c"}]
print(dedupe_by(rows, "id"), dedupe_by(rows, "id", keep="last"))

In [ ]:
assert dedupe_by([], "id") == [] and dedupe_by(rows[:1], "id") == rows[:1]     # empty, single
assert [r["v"] for r in dedupe_by(rows, "id")] == ["a", "b"]
assert [r["v"] for r in dedupe_by(rows, "id", keep="last")] == ["c", "b"]      # position of the FIRST, value of the LAST
users = [{"email": "Ann@x.com", "n": 1}, {"email": "ann@x.com ", "n": 2}, {"email": "bo@x.com", "n": 3}]
assert [u["n"] for u in dedupe_by(users, lambda u: u["email"].strip().casefold())] == [1, 3]   # normalised key
assert [u["n"] for u in dedupe_by(users, "email")] == [1, 2, 3]                 # raw key misses the duplicate
assert raises(KeyError, dedupe_by, [{"no_id": 1}], "id")                        # missing field: decide skip or raise
assert raises(TypeError, set, rows)                                             # why we need a key at all
assert raises(ValueError, dedupe_by, rows, "id", keep="middle")
print("dedupe_by: all tests passed")

**Complexity:** O(n) time and O(n) space. With `keep="last"`, the dict keeps each key at the position where it was *first* inserted: overwriting a value doesn't move the key. Normalise the key (strip, casefold) or you will miss real duplicates. To dedupe whole records, use `json.dumps(r, sort_keys=True)` as the key.

## Try it yourself

**1.** Flatten a nested dict into dotted keys, the inverse of `get_path`: `{"db": {"primary": {"port": 5432}}}` → `{"db.primary.port": 5432}`. Check that `get_path(CFG, key)` returns every flattened value.

In [ ]:
# Solution — try it yourself first, then run this
def flatten_dict(d, parent="", sep="."):
    flat = {}
    for k, v in d.items():
        key = f"{parent}{sep}{k}" if parent else str(k)
        if isinstance(v, dict) and v:
            flat.update(flatten_dict(v, key, sep))    # recurse into non-empty dicts
        else:
            flat[key] = v                             # lists and scalars are leaves
    return flat


flat = flatten_dict(CFG)
print(flat)
assert flatten_dict({}) == {} and flatten_dict({"a": {}}) == {"a": {}}
assert all(get_path(CFG, k) == v for k, v in flat.items())

**2.** Merge overlapping intervals: `[[1, 3], [2, 6], [8, 10], [15, 18]]` → `[[1, 6], [8, 10], [15, 18]]`. Hint: sort by start, then extend the last merged interval or start a new one.

In [ ]:
# Solution — try it yourself first, then run this
def merge_intervals(intervals):
    merged = []
    for start, end in sorted(intervals):
        if merged and start <= merged[-1][1]:         # overlaps (or touches) the last one
            merged[-1][1] = max(merged[-1][1], end)
        else:
            merged.append([start, end])
    return merged


assert merge_intervals([]) == [] and merge_intervals([[5, 7]]) == [[5, 7]]
assert merge_intervals([[1, 3], [2, 6], [8, 10], [15, 18]]) == [[1, 6], [8, 10], [15, 18]]
assert merge_intervals([[1, 4], [4, 5]]) == [[1, 5]]                     # touching
assert merge_intervals([[1, 10], [2, 3]]) == [[1, 10]]                   # contained
assert merge_intervals([[8, 9], [-3, 0], [-1, 2]]) == [[-3, 2], [8, 9]]  # unsorted, negatives
print(merge_intervals([[1, 3], [2, 6], [8, 10], [15, 18]]))

**3.** Product of the array except self, without division: `[1, 2, 3, 4]` → `[24, 12, 8, 6]`. Hint: running products from the left and from the right, as in Problem 5.

In [ ]:
# Solution — try it yourself first, then run this
def product_except_self(nums):
    out = [1] * len(nums)
    left = 1
    for i in range(len(nums)):                        # out[i] = product of everything left of i
        out[i] = left
        left *= nums[i]
    right = 1
    for i in reversed(range(len(nums))):              # times product of everything right of i
        out[i] *= right
        right *= nums[i]
    return out


assert product_except_self([]) == [] and product_except_self([5]) == [1]
assert product_except_self([1, 2, 3, 4]) == [24, 12, 8, 6]
assert product_except_self([0, 1, 2]) == [2, 0, 0]                      # division would break on the 0
assert product_except_self([-1, 2, -3]) == [-6, 3, -2]
print(product_except_self([1, 2, 3, 4]))

## Say it in an interview

**30-second answer for "how do you speed this up?":** *"The brute force compares every pair, which is O(n²). If I store what I've seen in a dict or set, each lookup is O(1) on average, so one pass is O(n) time for O(n) memory. If the input is sorted, two pointers get O(n) with O(1) memory. If they say 'in place', I use a write index."*

| Need | Tool | Complexity |
|---|---|---|
| Find a partner / membership | `dict` / `set` | O(1) average lookup |
| Count, top-k, intersection with counts | `Counter`, `heapq.nsmallest(k, ..., key=)` | O(n), O(m log k) |
| Group by a property | `defaultdict(list)` with a canonical key | O(n · key cost) |
| Sorted input, pairs or merges | two pointers, `heapq.merge` | O(n + m) |
| Range sums | prefix sums via `accumulate(..., initial=0)` | O(1) per query |
| Batches from a stream | generator + `islice`, or `itertools.batched` | O(batch) memory |

**Traps interviewers probe:** `{v: k}` loses keys on duplicate values; `Counter + Counter` drops zero and negative counts; dict `==` ignores order; `.get(...) or default` eats `0` and `False`; `zip` silently truncates ragged rows (`strict=True`); a generator's validation runs on the first `next()`; `sort(reverse=True)` stays stable; the 26-slot anagram key breaks on uppercase via negative indexing.

**Follow-ups:** "What if it doesn't fit in memory?" (stream it, count in chunks, merge sorted runs). "What if the dicts are huge and nested?" (a dotted-path getter, or pydantic models). "Which record wins on dedupe?" (first or last, plus a normalised key).

## Next

- [04 · Practical AI-engineer Python](04_practical_ai_engineer_python.ipynb): batching, caching, parsing and deduping for LLM systems.
- [Arrays and two pointers](../03_dsa/02_arrays_and_two_pointers.ipynb) · [Hashing and sliding window](../03_dsa/03_hashing_and_sliding_window.ipynb) · [Heaps and top-k](../03_dsa/10_heaps_and_top_k.ipynb) · [Greedy and intervals](../03_dsa/13_greedy_and_intervals.ipynb)
- [Dicts and sets](../01_python_basics/04_dicts_and_sets.ipynb) and [Comprehensions, iterators, generators](../01_python_basics/07_comprehensions_iterators_generators.ipynb)
- Theory: [python_basics course](../../python_basics/index.html) · [dsa_basics course](../../dsa_basics/index.html)